# MÓDULO 01 — Inventario de escenas Landsat 8/9

**Proyecto:** `lst-barranquilla-suhi-deep-learning`  
**Área:** Barranquilla, Colombia  
**Periodo operativo:** 2013–2025  
**WRS:** Path 9 / Row 52  
**Sensores:** Landsat 8 OLI/TIRS y Landsat 9 OLI-2/TIRS-2  
**Productos de referencia:**

- `LANDSAT/LC08/C02/T1_L2`
- `LANDSAT/LC09/C02/T1_L2`

## Objetivo

Construir una versión tabular, depurada y trazable del inventario de escenas Landsat 8/9 Collection 2 Level-2 disponibles para el path/row de Barranquilla durante el periodo operativo del estudio.

Este módulo no descarga imágenes, no modifica productos ráster y no calcula índices espectrales. Su función es documentar la disponibilidad temporal de escenas y generar tablas CSV que sirven como insumo para los módulos posteriores.

## Decisión metodológica sobre 2026

Aunque el inventario original incluía escenas de 2026, este año se excluye del periodo operativo porque la anualidad no está completa. Por tanto, el inventario usado en el flujo reproducible del manuscrito se restringe a 2013–2025.

## Entradas

Este notebook parte de un inventario previamente generado desde Google Earth Engine:

- Landsat 8 Collection 2 Level-2.
- Landsat 9 Collection 2 Level-2.
- WRS Path = 9.
- WRS Row = 52.

El uso de un CSV previamente generado evita repetir la autenticación y consulta a Earth Engine cuando el objetivo inmediato es consolidar el inventario definitivo del periodo 2013–2025.

## Salidas

Las salidas se generan localmente en Google Drive y corresponden conceptualmente a la carpeta del repositorio:

```text
data/scene_inventory/
```

Archivos generados:

```text
scene_inventory_landsat89_2013_2025.csv
annual_scene_count_landsat89_2013_2025.csv
monthly_scene_count_landsat89_2013_2025.csv
```

## Nota de reproducibilidad

El identificador del proyecto de Earth Engine no se fija en este notebook porque esta versión no consulta directamente Earth Engine. El inventario original debe conservarse como producto trazable de la consulta inicial y este módulo documenta la depuración temporal aplicada para el periodo operativo del estudio.


## 1. Importación de librerías

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## 2. Configuración del periodo operativo y del path/row

In [2]:
START_YEAR = 2013
END_YEAR = 2025

EXCLUDED_INCOMPLETE_YEARS = [2026]

SENSORS_EXPECTED = ["LC08", "LC09"]
WRS_PATH_EXPECTED = 9
WRS_ROW_EXPECTED = 52

print("Configuración del inventario")
print(f"Periodo operativo: {START_YEAR}–{END_YEAR}")
print(f"Años excluidos por anualidad incompleta: {EXCLUDED_INCOMPLETE_YEARS}")
print(f"Sensores esperados: {SENSORS_EXPECTED}")
print(f"WRS Path/Row esperado: {WRS_PATH_EXPECTED}/{WRS_ROW_EXPECTED}")


Configuración del inventario
Periodo operativo: 2013–2025
Años excluidos por anualidad incompleta: [2026]
Sensores esperados: ['LC08', 'LC09']
WRS Path/Row esperado: 9/52


## 3. Montaje de Google Drive

Esta celda es necesaria cuando el notebook se ejecuta en Google Colab. Si el notebook se ejecuta localmente, ajuste las rutas de entrada y salida en la siguiente sección.


In [3]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


## 4. Definición de rutas

El inventario original se lee desde la carpeta del proyecto en Google Drive. Las salidas ajustadas se guardan en una carpeta separada para preservar trazabilidad entre el inventario inicial y el inventario definitivo del periodo 2013–2025.


In [4]:
PROJECT_DIR = Path("/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling")

# Inventario original generado previamente desde Earth Engine.
IN_SCENE_CSV = PROJECT_DIR / "02_scene_inventory" / "scene_inventory_landsat89.csv"

# Carpeta local de salida para el inventario definitivo 2013–2025.
OUT_DIR = PROJECT_DIR / "02_scene_inventory_2013_2025"
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_SCENE_CSV = OUT_DIR / "scene_inventory_landsat89_2013_2025.csv"
OUT_ANNUAL_CSV = OUT_DIR / "annual_scene_count_landsat89_2013_2025.csv"
OUT_MONTHLY_CSV = OUT_DIR / "monthly_scene_count_landsat89_2013_2025.csv"

REPOSITORY_OUTPUT_DIR = "data/scene_inventory/"

print("Inventario original:")
print(IN_SCENE_CSV)

print("\nCarpeta local de salida:")
print(OUT_DIR)

print("\nCarpeta equivalente en el repositorio:")
print(REPOSITORY_OUTPUT_DIR)


Inventario original:
/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/02_scene_inventory/scene_inventory_landsat89.csv

Carpeta local de salida:
/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/02_scene_inventory_2013_2025

Carpeta equivalente en el repositorio:
data/scene_inventory/


## 5. Lectura del inventario original

In [5]:
if not IN_SCENE_CSV.exists():
    raise FileNotFoundError(
        "No se encontró el inventario original.\n"
        f"Ruta esperada:\n{IN_SCENE_CSV}\n\n"
        "Verifique que el archivo exista en Google Drive antes de continuar."
    )

df_original = pd.read_csv(IN_SCENE_CSV)

print("Inventario original cargado correctamente.")
print("Número de escenas en el inventario original:", len(df_original))

display(df_original.head(10))
display(df_original.tail(10))


Inventario original cargado correctamente.
Número de escenas en el inventario original: 380


,scene_id,sensor,date,year,month,doy,wrs_path,wrs_row,cloud_cover,cloud_cover_land,collection_category,processing_level,spacecraft_id
0,LC08_L2SP_009052_20130401_20200913_02_T1,LC08,2013-04-01,2013,4,91,9,52,8.20,21.50,T1,L2SP,LANDSAT_8
1,LC08_L2SP_009052_20130513_20200912_02_T1,LC08,2013-05-13,2013,5,133,9,52,16.20,24.11,T1,L2SP,LANDSAT_8
2,LC08_L2SP_009052_20130614_20200912_02_T1,LC08,2013-06-14,2013,6,165,9,52,17.20,15.33,T1,L2SP,LANDSAT_8
3,LC08_L2SP_009052_20130630_20200912_02_T1,LC08,2013-06-30,2013,6,181,9,52,64.25,15.68,T1,L2SP,LANDSAT_8
4,LC08_L2SP_009052_20130716_20200912_02_T1,LC08,2013-07-16,2013,7,197,9,52,73.55,52.27,T1,L2SP,LANDSAT_8
5,LC08_L2SP_009052_20130801_20200912_02_T1,LC08,2013-08-01,2013,8,213,9,52,57.53,26.26,T1,L2SP,LANDSAT_8
6,LC08_L2SP_009052_20130817_20200913_02_T1,LC08,2013-08-17,2013,8,229,9,52,21.81,32.07,T1,L2SP,LANDSAT_8
7,LC08_L2SP_009052_20130902_20200912_02_T1,LC08,2013-09-02,2013,9,245,9,52,11.01,27.56,T1,L2SP,LANDSAT_8
8,LC08_L2SP_009052_20130918_20200912_02_T1,LC08,2013-09-18,2013,9,261,9,52,32.34,70.99,T1,L2SP,LANDSAT_8
9,LC08_L2SP_009052_20131020_20200912_02_T1,LC08,2013-10-20,2013,10,293,9,52,10.65,23.25,T1,L2SP,LANDSAT_8


,scene_id,sensor,date,year,month,doy,wrs_path,wrs_row,cloud_cover,cloud_cover_land,collection_category,processing_level,spacecraft_id
370,LC08_L2SP_009052_20260226_20260306_02_T1,LC08,2026-02-26,2026,2,57,9,52,9.89,25.92,T1,L2SP,LANDSAT_8
371,LC09_L2SP_009052_20260306_20260307_02_T1,LC09,2026-03-06,2026,3,65,9,52,10.78,28.78,T1,L2SP,LANDSAT_9
372,LC08_L2SP_009052_20260314_20260328_02_T1,LC08,2026-03-14,2026,3,73,9,52,4.04,16.74,T1,L2SP,LANDSAT_8
373,LC09_L2SP_009052_20260322_20260324_02_T1,LC09,2026-03-22,2026,3,81,9,52,25.92,57.59,T1,L2SP,LANDSAT_9
374,LC08_L2SP_009052_20260330_20260407_02_T1,LC08,2026-03-30,2026,3,89,9,52,23.36,16.18,T1,L2SP,LANDSAT_8
375,LC09_L2SP_009052_20260407_20260408_02_T1,LC09,2026-04-07,2026,4,97,9,52,57.66,39.03,T1,L2SP,LANDSAT_9
376,LC08_L2SP_009052_20260415_20260424_02_T1,LC08,2026-04-15,2026,4,105,9,52,12.52,36.55,T1,L2SP,LANDSAT_8
377,LC09_L2SP_009052_20260423_20260424_02_T1,LC09,2026-04-23,2026,4,113,9,52,38.90,54.96,T1,L2SP,LANDSAT_9
378,LC08_L2SP_009052_20260501_20260506_02_T1,LC08,2026-05-01,2026,5,121,9,52,12.88,14.70,T1,L2SP,LANDSAT_8
379,LC09_L2SP_009052_20260509_20260510_02_T1,LC09,2026-05-09,2026,5,129,9,52,5.90,7.24,T1,L2SP,LANDSAT_9


## 6. Validación de columnas requeridas

El inventario debe contener los campos mínimos necesarios para documentar la escena, el sensor, la fecha de adquisición, el path/row, la nubosidad y el nivel de procesamiento.


In [6]:
required_cols = [
    "scene_id",
    "sensor",
    "date",
    "year",
    "month",
    "doy",
    "wrs_path",
    "wrs_row",
    "cloud_cover",
    "cloud_cover_land",
    "collection_category",
    "processing_level",
    "spacecraft_id",
]

missing_cols = [col for col in required_cols if col not in df_original.columns]

if missing_cols:
    raise ValueError(
        "El inventario no contiene las columnas requeridas:\n"
        + "\n".join(missing_cols)
    )

print("Validación de columnas completada.")
print("Todas las columnas requeridas están presentes.")


Validación de columnas completada.
Todas las columnas requeridas están presentes.


## 7. Normalización de tipos de datos

Esta etapa reduce ambigüedades antes de aplicar filtros temporales y controles de consistencia.


In [7]:
df_original = df_original.copy()

df_original["date"] = pd.to_datetime(df_original["date"], errors="coerce")
df_original["year"] = pd.to_numeric(df_original["year"], errors="coerce").astype("Int64")
df_original["month"] = pd.to_numeric(df_original["month"], errors="coerce").astype("Int64")
df_original["doy"] = pd.to_numeric(df_original["doy"], errors="coerce").astype("Int64")
df_original["wrs_path"] = pd.to_numeric(df_original["wrs_path"], errors="coerce").astype("Int64")
df_original["wrs_row"] = pd.to_numeric(df_original["wrs_row"], errors="coerce").astype("Int64")

df_original["sensor"] = df_original["sensor"].astype(str)
df_original["scene_id"] = df_original["scene_id"].astype(str)

invalid_dates = int(df_original["date"].isna().sum())

if invalid_dates > 0:
    raise ValueError(
        f"Se encontraron {invalid_dates} registros con fecha inválida. "
        "Revise la columna 'date'."
    )

print("Normalización de tipos completada.")
display(df_original.dtypes.to_frame("dtype"))


Normalización de tipos completada.


,dtype
scene_id,object
sensor,object
date,datetime64[ns]
year,Int64
month,Int64
doy,Int64
wrs_path,Int64
wrs_row,Int64
cloud_cover,float64
cloud_cover_land,float64


## 8. Revisión del inventario original

Antes del filtrado definitivo se reportan años, sensores y path/row presentes en el inventario original.


In [8]:
print("Años presentes en el inventario original:")
print(sorted(df_original["year"].dropna().astype(int).unique().tolist()))

print("\nSensores presentes en el inventario original:")
print(sorted(df_original["sensor"].dropna().unique().tolist()))

print("\nPath/Row presentes en el inventario original:")
pathrow_summary = (
    df_original[["wrs_path", "wrs_row"]]
    .drop_duplicates()
    .sort_values(["wrs_path", "wrs_row"])
    .reset_index(drop=True)
)
display(pathrow_summary)

print("\nConteo original por año y sensor:")
original_annual_counts = (
    df_original.groupby(["year", "sensor"])
    .size()
    .reset_index(name="n_scenes")
    .sort_values(["year", "sensor"])
    .reset_index(drop=True)
)
display(original_annual_counts)


Años presentes en el inventario original:
[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

Sensores presentes en el inventario original:
['LC08', 'LC09']

Path/Row presentes en el inventario original:


,wrs_path,wrs_row
0,9,52



Conteo original por año y sensor:


,year,sensor,n_scenes
0,2013,LC08,14
1,2014,LC08,21
2,2015,LC08,23
3,2016,LC08,23
4,2017,LC08,21
5,2018,LC08,22
6,2019,LC08,22
7,2020,LC08,22
8,2021,LC08,23
9,2021,LC09,2


## 9. Filtrado al periodo operativo 2013–2025

El periodo operativo se restringe a anualidades completas. Las escenas de 2026 no se incorporan al inventario definitivo del estudio.


In [9]:
df = df_original[
    (df_original["year"] >= START_YEAR) &
    (df_original["year"] <= END_YEAR)
].copy()

df = df.sort_values(["year", "date", "sensor"]).reset_index(drop=True)

print("Filtrado temporal completado.")
print("Número de escenas original:", len(df_original))
print("Número de escenas en periodo operativo:", len(df))
print("Número de escenas excluidas:", len(df_original) - len(df))

print("\nAños presentes después del ajuste:")
print(sorted(df["year"].dropna().astype(int).unique().tolist()))


Filtrado temporal completado.
Número de escenas original: 380
Número de escenas en periodo operativo: 364
Número de escenas excluidas: 16

Años presentes después del ajuste:
[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


## 10. Controles de consistencia

Los controles verifican que el inventario final sea consistente con el diseño metodológico: periodo 2013–2025, sensores Landsat 8/9 y path/row 9/52.


In [10]:
invalid_years = sorted(
    df.loc[df["year"] > END_YEAR, "year"].dropna().astype(int).unique().tolist()
)

if invalid_years:
    raise ValueError(
        f"El inventario contiene años posteriores al periodo operativo: {invalid_years}"
    )

excluded_years_present = sorted(
    set(df["year"].dropna().astype(int).unique().tolist()).intersection(EXCLUDED_INCOMPLETE_YEARS)
)

if excluded_years_present:
    raise ValueError(
        "El inventario contiene años excluidos por anualidad incompleta: "
        f"{excluded_years_present}"
    )

unexpected_sensors = sorted(set(df["sensor"].unique().tolist()) - set(SENSORS_EXPECTED))

if unexpected_sensors:
    raise ValueError(
        f"Se encontraron sensores no esperados: {unexpected_sensors}"
    )

unexpected_pathrow = df[
    (df["wrs_path"] != WRS_PATH_EXPECTED) |
    (df["wrs_row"] != WRS_ROW_EXPECTED)
]

if len(unexpected_pathrow) > 0:
    raise ValueError(
        "Se encontraron escenas fuera del WRS Path/Row esperado "
        f"{WRS_PATH_EXPECTED}/{WRS_ROW_EXPECTED}."
    )

print("Controles de consistencia completados.")
print("El inventario final es coherente con el periodo, sensores y path/row definidos.")


Controles de consistencia completados.
El inventario final es coherente con el periodo, sensores y path/row definidos.


## 11. Reordenamiento de columnas

Se conserva una estructura tabular estable para facilitar el uso del inventario en módulos posteriores.


In [11]:
cols = [
    "scene_id",
    "sensor",
    "date",
    "year",
    "month",
    "doy",
    "wrs_path",
    "wrs_row",
    "cloud_cover",
    "cloud_cover_land",
    "collection_category",
    "processing_level",
    "spacecraft_id",
]

extra_cols = [col for col in df.columns if col not in cols]
final_cols = cols + extra_cols

df = df[final_cols].copy()
df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")

print("Columnas del inventario final:")
print(df.columns.tolist())


Columnas del inventario final:
['scene_id', 'sensor', 'date', 'year', 'month', 'doy', 'wrs_path', 'wrs_row', 'cloud_cover', 'cloud_cover_land', 'collection_category', 'processing_level', 'spacecraft_id']


## 12. Resúmenes temporales

Se generan resúmenes anuales y mensuales por sensor para documentar la disponibilidad temporal de escenas.


In [12]:
annual_counts = (
    df.groupby(["year", "sensor"])
    .size()
    .reset_index(name="n_scenes")
    .sort_values(["year", "sensor"])
    .reset_index(drop=True)
)

annual_total = (
    df.groupby("year")
    .size()
    .reset_index(name="n_scenes_total")
    .sort_values("year")
    .reset_index(drop=True)
)

monthly_counts = (
    df.groupby(["year", "month", "sensor"])
    .size()
    .reset_index(name="n_scenes")
    .sort_values(["year", "month", "sensor"])
    .reset_index(drop=True)
)

print("Resumen anual por sensor:")
display(annual_counts)

print("\nResumen anual total:")
display(annual_total)

print("\nResumen mensual por sensor:")
display(monthly_counts)


Resumen anual por sensor:


,year,sensor,n_scenes
0,2013,LC08,14
1,2014,LC08,21
2,2015,LC08,23
3,2016,LC08,23
4,2017,LC08,21
5,2018,LC08,22
6,2019,LC08,22
7,2020,LC08,22
8,2021,LC08,23
9,2021,LC09,2



Resumen anual total:


,year,n_scenes_total
0,2013,14
1,2014,21
2,2015,23
3,2016,23
4,2017,21
5,2018,22
6,2019,22
7,2020,22
8,2021,25
9,2022,42



Resumen mensual por sensor:


,year,month,sensor,n_scenes
0,2013,4,LC08,1
1,2013,5,LC08,1
2,2013,6,LC08,2
3,2013,7,LC08,1
4,2013,8,LC08,2
...,...,...,...,...
196,2025,10,LC09,2
197,2025,11,LC08,2
198,2025,11,LC09,2
199,2025,12,LC08,2


## 13. Guardado de productos

Las tablas se guardan como CSV para mantener trazabilidad y permitir su inspección independiente del notebook.


In [13]:
df.to_csv(OUT_SCENE_CSV, index=False)
annual_counts.to_csv(OUT_ANNUAL_CSV, index=False)
monthly_counts.to_csv(OUT_MONTHLY_CSV, index=False)

print("Archivos CSV generados:")
print(OUT_SCENE_CSV)
print(OUT_ANNUAL_CSV)
print(OUT_MONTHLY_CSV)


Archivos CSV generados:
/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/02_scene_inventory_2013_2025/scene_inventory_landsat89_2013_2025.csv
/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/02_scene_inventory_2013_2025/annual_scene_count_landsat89_2013_2025.csv
/content/drive/MyDrive/Barranquilla_LST_2013_2026_Modeling/02_scene_inventory_2013_2025/monthly_scene_count_landsat89_2013_2025.csv


## 14. Reporte final del inventario

Esta sección resume los elementos principales del inventario definitivo que será usado por los módulos posteriores.


In [14]:
print("Inventario Landsat 8/9 — Barranquilla, Colombia")
print(f"Periodo operativo: {START_YEAR}–{END_YEAR}")
print(f"Path/Row: {WRS_PATH_EXPECTED}/{WRS_ROW_EXPECTED}")
print(f"Sensores incluidos: {sorted(df['sensor'].unique().tolist())}")
print(f"Número de escenas original: {len(df_original)}")
print(f"Número de escenas en inventario final: {len(df)}")
print(f"Años presentes: {sorted(df['year'].astype(int).unique().tolist())}")

print("\nProductos generados:")
print(f"- {OUT_SCENE_CSV.name}")
print(f"- {OUT_ANNUAL_CSV.name}")
print(f"- {OUT_MONTHLY_CSV.name}")

print("\nResumen anual por sensor:")
display(annual_counts)

print("\nPrimeras 20 escenas del inventario final:")
display(df.head(20))

print("\nÚltimas 20 escenas del inventario final:")
display(df.tail(20))


Inventario Landsat 8/9 — Barranquilla, Colombia
Periodo operativo: 2013–2025
Path/Row: 9/52
Sensores incluidos: ['LC08', 'LC09']
Número de escenas original: 380
Número de escenas en inventario final: 364
Años presentes: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Productos generados:
- scene_inventory_landsat89_2013_2025.csv
- annual_scene_count_landsat89_2013_2025.csv
- monthly_scene_count_landsat89_2013_2025.csv

Resumen anual por sensor:


,year,sensor,n_scenes
0,2013,LC08,14
1,2014,LC08,21
2,2015,LC08,23
3,2016,LC08,23
4,2017,LC08,21
5,2018,LC08,22
6,2019,LC08,22
7,2020,LC08,22
8,2021,LC08,23
9,2021,LC09,2



Primeras 20 escenas del inventario final:


,scene_id,sensor,date,year,month,doy,wrs_path,wrs_row,cloud_cover,cloud_cover_land,collection_category,processing_level,spacecraft_id
0,LC08_L2SP_009052_20130401_20200913_02_T1,LC08,2013-04-01,2013,4,91,9,52,8.20,21.50,T1,L2SP,LANDSAT_8
1,LC08_L2SP_009052_20130513_20200912_02_T1,LC08,2013-05-13,2013,5,133,9,52,16.20,24.11,T1,L2SP,LANDSAT_8
2,LC08_L2SP_009052_20130614_20200912_02_T1,LC08,2013-06-14,2013,6,165,9,52,17.20,15.33,T1,L2SP,LANDSAT_8
3,LC08_L2SP_009052_20130630_20200912_02_T1,LC08,2013-06-30,2013,6,181,9,52,64.25,15.68,T1,L2SP,LANDSAT_8
4,LC08_L2SP_009052_20130716_20200912_02_T1,LC08,2013-07-16,2013,7,197,9,52,73.55,52.27,T1,L2SP,LANDSAT_8
5,LC08_L2SP_009052_20130801_20200912_02_T1,LC08,2013-08-01,2013,8,213,9,52,57.53,26.26,T1,L2SP,LANDSAT_8
6,LC08_L2SP_009052_20130817_20200913_02_T1,LC08,2013-08-17,2013,8,229,9,52,21.81,32.07,T1,L2SP,LANDSAT_8
7,LC08_L2SP_009052_20130902_20200912_02_T1,LC08,2013-09-02,2013,9,245,9,52,11.01,27.56,T1,L2SP,LANDSAT_8
8,LC08_L2SP_009052_20130918_20200912_02_T1,LC08,2013-09-18,2013,9,261,9,52,32.34,70.99,T1,L2SP,LANDSAT_8
9,LC08_L2SP_009052_20131020_20200912_02_T1,LC08,2013-10-20,2013,10,293,9,52,10.65,23.25,T1,L2SP,LANDSAT_8



Últimas 20 escenas del inventario final:


,scene_id,sensor,date,year,month,doy,wrs_path,wrs_row,cloud_cover,cloud_cover_land,collection_category,processing_level,spacecraft_id
344,LC09_L2SP_009052_20250725_20250727_02_T1,LC09,2025-07-25,2025,7,206,9,52,14.84,30.58,T1,L2SP,LANDSAT_9
345,LC08_L2SP_009052_20250802_20250814_02_T1,LC08,2025-08-02,2025,8,214,9,52,25.48,29.63,T1,L2SP,LANDSAT_8
346,LC09_L2SP_009052_20250810_20250811_02_T1,LC09,2025-08-10,2025,8,222,9,52,74.76,89.04,T1,L2SP,LANDSAT_9
347,LC08_L2SP_009052_20250818_20250826_02_T1,LC08,2025-08-18,2025,8,230,9,52,27.76,38.03,T1,L2SP,LANDSAT_8
348,LC09_L2SP_009052_20250826_20250902_02_T1,LC09,2025-08-26,2025,8,238,9,52,24.71,29.45,T1,L2SP,LANDSAT_9
349,LC08_L2SP_009052_20250903_20250912_02_T1,LC08,2025-09-03,2025,9,246,9,52,12.63,27.67,T1,L2SP,LANDSAT_8
350,LC09_L2SP_009052_20250911_20250915_02_T1,LC09,2025-09-11,2025,9,254,9,52,13.35,13.58,T1,L2SP,LANDSAT_9
351,LC08_L2SP_009052_20250919_20250929_02_T1,LC08,2025-09-19,2025,9,262,9,52,38.13,61.82,T1,L2SP,LANDSAT_8
352,LC09_L2SP_009052_20250927_20250928_02_T1,LC09,2025-09-27,2025,9,270,9,52,40.35,39.38,T1,L2SP,LANDSAT_9
353,LC08_L2SP_009052_20251005_20251115_02_T1,LC08,2025-10-05,2025,10,278,9,52,20.21,12.16,T1,L2SP,LANDSAT_8
